In [ ]:
# ===== A0. Setup: analysis on CPU from the saved predictions of the main pipeline run =====
# Project: Label-free recalibration of skin cancer referral thresholds under site shift
# Author: Arshad Saeed, Bay Atlantic University
import os, sys, glob, json, importlib, numpy as np, pandas as pd
cands = [p for p in glob.glob('/kaggle/input/**/SkinCancer_Referral_Q1_2026', recursive=True) if os.path.isdir(p + '/models')]
INP = max(cands, key=lambda p: len(os.listdir(p + '/models'))); print('input:', INP)
WORK = '/kaggle/working/SkinCancer_Referral_Q1_2026'; RES = f'{WORK}/results'; os.makedirs(RES, exist_ok=True)
os.makedirs(f'{WORK}/notebooks', exist_ok=True)
sys.path.insert(0, f'{WORK}/notebooks')


In [ ]:
%%writefile /kaggle/working/SkinCancer_Referral_Q1_2026/notebooks/referral_lib.py
"""
referral_lib.py — melanoma-safe referral and label-free site recalibration.
Project: Label-free site recalibration for melanoma-safe referral (multi-center dermoscopy)
Author: Arshad Saeed, Bay Atlantic University

Policy: a case is AUTO-CLEARED (not referred) when its malignancy score s < tau, otherwise REFERRED.
Safety target: P(s < tau | malignant) <= alpha  (malignant miss rate on the deployment site).

Threshold methods (all except ORACLE use NO target labels):
  SRC      split-conformal class-conditional threshold fitted on source calibration data only
  WCP      weighted conformal (Tibshirani et al., 2019): calibration malignant scores re-weighted by
           estimated density ratio p_target(x)/p_source(x) from a cross-fitted domain classifier on embeddings.
           Per-test-case threshold; if the test point carries too much weight the case is referred
           ("certify or refuse").
  QTC      quantile threshold matching (Yilmaz & Heckel): keep the source auto-clear fraction on the target
  QTC_LS   QTC with the auto-clear fraction rescaled by the EM-estimated target benign prevalence
  ORACLE_n uses n labeled target malignant cases (few-label reference, not label-free)
"""
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

CLASSES = ['MEL', 'NV', 'BCC', 'AKIEC', 'BKL', 'DF', 'VASC']
MAL_IDX = [0, 2, 3]   # MEL, BCC, AKIEC


def softmax(z):
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)


def fit_temperature(logits, y, grid=np.linspace(0.5, 5, 46)):
    best, bt = np.inf, 1.0
    for t in grid:
        p = softmax(logits / t)
        nll = -np.log(p[np.arange(len(y)), y] + 1e-12).mean()
        if nll < best:
            best, bt = nll, t
    return bt


def mal_score(probs):
    return probs[:, MAL_IDX].sum(1)


# ---------------------------------------------------------------- thresholds
def conformal_lower(scores_mal, alpha):
    """Largest tau with P(s_new < tau) <= alpha under exchangeability (finite-sample)."""
    s = np.sort(scores_mal)
    k = int(np.floor(alpha * (len(s) + 1)))
    return -np.inf if k < 1 else s[k - 1]


def weighted_lower(scores_mal, w_cal, w_test, alpha):
    """Per-test weighted conformal lower threshold (Tibshirani et al. 2019, lower tail).
    The test point's own mass sits at -inf, so tau = -inf (refer) when that mass alone exceeds alpha."""
    order = np.argsort(scores_mal)
    s, w = scores_mal[order], w_cal[order]
    W = w.sum()
    taus = np.empty(len(w_test))
    cw = np.cumsum(w)
    for i, wt in enumerate(w_test):
        tot = W + wt
        # mass strictly below candidate s_j (exclusive) = wt + cw[j-1]
        below = (wt + np.concatenate([[0.0], cw[:-1]])) / tot
        ok = np.where(below <= alpha)[0]
        taus[i] = -np.inf if (wt / tot > alpha or len(ok) == 0) else s[ok[-1]]
    return taus


def density_ratio(emb_src, emb_tgt, n_pca=64, seed=0, clip=50.0):
    """Cross-fitted domain classifier -> w(x) = p_t(x)/p_s(x) for source AND target points.
    Returns (w_src, w_tgt, domain_auc)."""
    X = np.vstack([emb_src, emb_tgt]).astype(np.float32)
    d = np.r_[np.zeros(len(emb_src)), np.ones(len(emb_tgt))]
    X = StandardScaler().fit_transform(X)
    X = PCA(n_components=min(n_pca, X.shape[1], len(X) - 1), random_state=seed).fit_transform(X)
    p = np.zeros(len(X))
    for tr, te in StratifiedKFold(2, shuffle=True, random_state=seed).split(X, d):
        clf = LogisticRegression(C=0.1, max_iter=2000).fit(X[tr], d[tr])
        p[te] = clf.predict_proba(X[te])[:, 1]
    auc = roc_auc_score(d, p)
    p = np.clip(p, 1e-4, 1 - 1e-4)
    w = (p / (1 - p)) * (len(emb_src) / len(emb_tgt))
    w = np.clip(w, 0, clip)
    return w[:len(emb_src)], w[len(emb_src):], auc


def em_prior(probs_t, prior_s, iters=200, tol=1e-6):
    """Saerens et al. (2002) EM estimate of the target class prior (label shift), no target labels."""
    prior_s = np.clip(np.asarray(prior_s, float), 1e-6, None); prior_s = prior_s / prior_s.sum()
    pi = prior_s.copy()
    for _ in range(iters):
        q = probs_t * (pi / prior_s)
        q /= q.sum(1, keepdims=True)
        new = q.mean(0)
        if np.abs(new - pi).max() < tol:
            pi = new
            break
        pi = new
    return pi


# ---------------------------------------------------------------- one site
def evaluate_site(src, tgt, alphas=(0.05, 0.10), oracle_ns=(20, 50), seed=0):
    """
    src: dict with  s_cal (all cal scores), y_cal_mal (0/1), emb_cal, probs_cal, prior_s
    tgt: dict with  s, y_mal (0/1, used ONLY for evaluation and ORACLE), is_mel (0/1), emb, probs, lesion (ids)
    Returns list of result rows (one per alpha x method).
    """
    rng = np.random.default_rng(seed)
    s_mal_src = src['s_cal'][src['y_cal_mal'] == 1]
    w_src, w_tgt, dom_auc = density_ratio(src['emb_cal'], tgt['emb'], seed=seed)
    w_mal = w_src[src['y_cal_mal'] == 1]
    n_eff = w_mal.sum() ** 2 / (w_mal ** 2).sum()
    pi_t = em_prior(tgt['probs'], src['prior_s'])
    mal_prev_s = src['prior_s'][MAL_IDX].sum()
    mal_prev_t_hat = pi_t[MAL_IDX].sum()
    rows = []
    for a in alphas:
        th = {}
        tau_src = conformal_lower(s_mal_src, a)
        th['SRC'] = np.full(len(tgt['s']), tau_src)
        th['WCP'] = weighted_lower(s_mal_src, w_mal, w_tgt, a)
        clear_frac_src = np.mean(src['s_cal'] < tau_src)
        th['QTC'] = np.full(len(tgt['s']), np.quantile(tgt['s'], clear_frac_src) if clear_frac_src > 0 else -np.inf)
        cf = np.clip(clear_frac_src * (1 - mal_prev_t_hat) / (1 - mal_prev_s), 0, 1)
        th['QTC_LS'] = np.full(len(tgt['s']), np.quantile(tgt['s'], cf) if cf > 0 else -np.inf)
        mal_idx = np.where(tgt['y_mal'] == 1)[0]
        for n in oracle_ns:
            if len(mal_idx) > n:
                pick = rng.choice(mal_idx, n, replace=False)
                t = conformal_lower(tgt['s'][pick], a)
                th[f'ORACLE_{n}'] = np.full(len(tgt['s']), t)
        for m, tau in th.items():
            cleared = tgt['s'] < tau
            r = summarize(cleared, tgt)
            r.update(dict(alpha=a, method=m, domain_auc=dom_auc, n_eff_mal=n_eff,
                          est_mal_prev=mal_prev_t_hat, true_mal_prev=tgt['y_mal'].mean()))
            rows.append(r)
    return rows


def summarize(cleared, tgt):
    y, mel = tgt['y_mal'] == 1, tgt['is_mel'] == 1
    return dict(
        n=len(y), n_mal=int(y.sum()), n_mel=int(mel.sum()),
        mal_miss=float(cleared[y].mean()) if y.any() else np.nan,
        mel_miss=float(cleared[mel].mean()) if mel.any() else np.nan,
        referral_rate=float((~cleared).mean()),
        auto_clear_npv=float((~y[cleared]).mean()) if cleared.any() else np.nan,
    )


def lesion_bootstrap(cleared, tgt, B=1000, seed=0):
    """95% CI for mal_miss and referral_rate, resampling lesions."""
    rng = np.random.default_rng(seed)
    les = np.asarray(tgt['lesion'])
    uniq, inv = np.unique(les, return_inverse=True)
    groups = [np.where(inv == g)[0] for g in range(len(uniq))]
    mm, rr = [], []
    for _ in range(B):
        idx = np.concatenate([groups[g] for g in rng.integers(0, len(groups), len(groups))])
        y = tgt['y_mal'][idx] == 1
        c = cleared[idx]
        if y.any():
            mm.append(c[y].mean())
        rr.append((~c).mean())
    return np.percentile(mm, [2.5, 97.5]), np.percentile(rr, [2.5, 97.5])


In [ ]:
# ===== A1. Load manifests + sanitize non-finite predictions (fp16 overflow) =====
import referral_lib; importlib.reload(referral_lib); from referral_lib import *
man = {n: pd.read_csv(f'{INP}/manifests/{n}.csv') for n in ['ham10000', 'bcn20000', 'milk10k', 'padufes20']}
C2I = {c: i for i, c in enumerate(CLASSES)}
runs = sorted(glob.glob(f'{INP}/models/*__split_s*/preds.npz'))
leaky = sorted(glob.glob(f'{INP}/models/*__leaky_s*/preds.npz'))
# EfficientNet-B3 predictions recomputed in fp32 (separate run) replace the fp16 ones when available
FP32 = {f.split('/')[-2]: f for f in glob.glob('/kaggle/input/**/effnet_fp32/models/*/preds.npz', recursive=True)}
runs = [FP32.get(f.split('/')[-2], f) for f in runs]; leaky = [FP32.get(f.split('/')[-2], f) for f in leaky]
print('fp32 replacements:', sorted(FP32))
print(len(runs), 'clean runs;', len(leaky), 'leaky runs')
NONFINITE = []
FORCE_REFER = np.array([20., 0, 0, 0, 0, 0, 0], np.float32)   # MEL logit high -> s(x)~1 -> always referred (conservative)
def load_preds(f):
    tag = f.split('/')[-2]; P = {}
    with np.load(f) as Z:
        for k in Z.files:
            v = Z[k].astype(np.float32)
            if v.ndim == 2 and not np.isfinite(v).all():
                rows_nan = np.isnan(v).any(1); rows_inf = np.isinf(v).any(1) & ~rows_nan
                NONFINITE.append(dict(run=tag, array=k, n_rows=len(v), rows_nan=int(rows_nan.sum()), rows_inf=int(rows_inf.sum())))
                if k.endswith('_logits'):
                    v = np.nan_to_num(v, nan=0.0, posinf=60.0, neginf=-60.0); v[rows_nan] = FORCE_REFER
                else:
                    v = np.nan_to_num(v, nan=0.0, posinf=6e4, neginf=-6e4)
            P[k] = v
    return P
_ = [load_preds(f) for f in runs + leaky]
NF = pd.DataFrame(NONFINITE); NF.to_csv(f'{RES}/nonfinite_report.csv', index=False)
print('non-finite arrays:', len(NF)); print(NF.to_string() if len(NF) else 'none')
NONFINITE.clear()


In [ ]:
# ===== LESION-LEVEL SENSITIVITY: refer a lesion if ANY of its images is referred =====
# Primary analysis is per image. Here thresholds are unchanged, but decisions are aggregated per lesion:
# a lesion counts as auto-cleared only if ALL its images are auto-cleared (conservative clinical rule).
import hashlib
EXT = ['bcn20000', 'milk10k', 'padufes20']
rows = []
for f in runs:
    tag = f.split('/')[-2]; arch, split = tag.split('__'); seed = int(split[-2:])
    P = load_preds(f); ham = man['ham10000']
    cal = (ham[split] == 'val').values
    T = fit_temperature(P['ham10000_logits'][cal], ham['label7'][cal].map(C2I).values)
    pc = softmax(P['ham10000_logits'][cal] / T); s_cal = mal_score(pc); y_cal = ham['malignant'][cal].values
    s_mal = s_cal[y_cal == 1]
    for n in EXT:
        d = man[n]; pt = softmax(P[f'{n}_logits'] / T); s = mal_score(pt)
        w_src, w_tgt, _ = density_ratio(P['ham10000_emb'][cal].astype(np.float32), P[f'{n}_emb'].astype(np.float32), seed=seed)
        rng = np.random.default_rng(seed); mal_idx = np.where(d['malignant'].values == 1)[0]
        for a in (0.05, 0.10):
            tau = {'SRC': np.full(len(s), conformal_lower(s_mal, a)),
                   'WCP': weighted_lower(s_mal, w_src[y_cal == 1], w_tgt, a)}
            for k in (20, 50):   # same random draws as the main analysis
                pick = rng.choice(mal_idx, k, replace=False)
                if k == 50: tau['ORACLE_50'] = np.full(len(s), conformal_lower(s[pick], a))
            for m, t in tau.items():
                clr = s < t
                L = pd.DataFrame({'lesion': d['lesion_id'].values, 'mal': d['malignant'].values, 'clr': clr}) \
                      .groupby('lesion').agg(mal=('mal', 'max'), clr=('clr', 'all'))
                rows.append(dict(model=arch, seed=seed, site=n, alpha=a, method=m,
                                 n_img=len(d), n_les=len(L),
                                 img_miss=clr[d['malignant'].values == 1].mean(), img_ref=(~clr).mean(),
                                 les_miss=L.clr[L.mal == 1].mean(), les_ref=(~L.clr).mean()))
LS = pd.DataFrame(rows); LS['img_break'] = LS.img_miss > LS.alpha; LS['les_break'] = LS.les_miss > LS.alpha
LS.to_csv(f'{RES}/lesion_level_sensitivity_raw.csv', index=False)
S = LS.groupby(['alpha', 'site', 'method'])[['n_img', 'n_les', 'img_miss', 'les_miss', 'img_ref', 'les_ref', 'img_break', 'les_break']].mean()
S.round(3).to_csv(f'{RES}/lesion_level_sensitivity_summary.csv')
P_ = LS.groupby(['alpha', 'method'])[['img_miss', 'les_miss', 'img_ref', 'les_ref', 'img_break', 'les_break']].mean()
print('LESION_SUMMARY_START'); print(S.round(3).to_csv()); print('POOLED'); print(P_.round(3).to_csv()); print('LESION_SUMMARY_END')
# checksums of the split / manifest files
cs = []
for n in ['ham10000', 'bcn20000', 'milk10k', 'padufes20']:
    p = f'{INP}/manifests/{n}.csv'; cs.append(dict(file=f'{n}.csv', rows=len(man[n]), sha256=hashlib.sha256(open(p, 'rb').read()).hexdigest()))
    man[n].to_csv(f'{WORK}/{n}_manifest.csv', index=False)
print('CHECKSUMS_START'); print(pd.DataFrame(cs).to_csv(index=False)); print('CHECKSUMS_END')
print('HAM split columns:', [c for c in man['ham10000'].columns if c.startswith(('split_', 'leaky_'))])
